# 05 — Figures and Bootstrap CI

**Pipeline stage:** turning the summary tables from notebook 04 into the actual figures referenced by
the paper.

Two figure-generating scripts, plus one narrower plotting script:

- `scripts/generate_original_study_figures.py` — the full 16-figure diagnostic suite (data coverage,
  wind/solar balance, gain-vs-wind-dominance scatter, per-country gain heatmap, resolution-drift
  boxplots, compute/fidelity frontier, calibration curves, and more). Meant for the authors' own
  diagnostic use, not necessarily all for publication.
- `scripts/generate_paper_placeholder_figures.py` — exactly the 7 figures assembled for the paper
  itself (pipeline diagram, country choropleth, capacity-weighting map, gain scatter, held-out
  time-series example, resolution/weighting sensitivity, bootstrap CIs).
- `scripts/plot_post_covid_bootstrap.py` — a focused 2×2 panel of block-bootstrap confidence intervals,
  one panel per model.

This notebook's job is orientation — showing what each script produces and why — not regenerating the
figures here, since that needs the full staged dataset from notebooks 01–04.

In [1]:
!python ../scripts/generate_original_study_figures.py --help

usage: generate_original_study_figures.py [-h]
                                          [--formats {png,pdf} [{png,pdf} ...]]
                                          [--dpi DPI] [--skip-refit]
                                          [--example-days EXAMPLE_DAYS]
                                          [--only-capacity-weighting]

Generate a diagnostic figure suite for the original prediction study. This
script intentionally excludes every HPC-scheduling result. It reads the
existing Energy-Charts/ERA5 inputs and spatial-resolution model results, then
creates figures that answer four questions: 1. What data are available, and
how different are the 19 electricity systems? 2. Does weather improve held-out
prediction beyond calendar features? 3. Is weather-added gain consistently
associated with wind-minus-solar share? 4. How sensitive is that conclusion to
spatial weighting and resolution? The default run also refits two small
diagnostics with the same chronological 80/20 protocol:

In [2]:
!python ../scripts/generate_paper_placeholder_figures.py --help

usage: generate_paper_placeholder_figures.py [-h]
                                             [--formats {png,pdf} [{png,pdf} ...]]
                                             [--dpi DPI]
                                             [--example-days EXAMPLE_DAYS]

Generate Figures 1-7 requested by the original prediction paper. Outputs are
written to ``figures/paper_placeholders`` as both PNG and PDF. The analysis
uses the corrected 0.25-degree capacity-weighted model results unless a figure
explicitly examines spatial resolution or uncertainty. Figure 1: analysis
pipeline Figure 2: study-country map colored by wind-minus-solar share Figure
3: Denmark capacity-weighting map Figure 4: weather-added gain versus wind-
minus-solar share Figure 5: Denmark held-out prediction time series Figure 6:
resolution and weighting sensitivity Figure 7: weekly-block bootstrap
confidence intervals All paths are anchored to this file rather than the
current working directory.

options:
  -h, --help    

In [3]:
!python ../scripts/plot_post_covid_bootstrap.py --help

usage: plot_post_covid_bootstrap.py [-h] [--input INPUT]
                                    [--output-dir OUTPUT_DIR]
                                    [--metric-family {score,error}]
                                    [--formats {png,pdf} [{png,pdf} ...]]
                                    [--dpi DPI]

Plot four-model weekly block-bootstrap intervals.

options:
  -h, --help            show this help message and exit
  --input INPUT
  --output-dir OUTPUT_DIR
  --metric-family {score,error}
  --formats {png,pdf} [{png,pdf} ...]
  --dpi DPI


## What each script needs, and where its output goes

| Script | Needs | Writes to |
|---|---|---|
| `generate_original_study_figures.py` | per-country energy + weather CSVs (notebooks 01–02), canonical model results (notebook 04) | `figures/original_study_diagnostics/`, `figures/research_paper_figure_addons/`, `results/diagnostics/original_study/` |
| `generate_paper_placeholder_figures.py` | the above, plus a Natural Earth country-boundary shapefile (bundled with the `pyogrio` package — see the README's note about this being a fragile dependency) | `figures/paper_placeholders/`, `results/manuscript/placeholder_data/` |
| `plot_post_covid_bootstrap.py` | `results/post_covid_spatial_resolution/block_bootstrap/block_bootstrap_country_metrics.csv` (from `bootstrap_post_covid_all_models.py` in notebook 04) | `figures/original_study_diagnostics/15_block_bootstrap_all_models_<family>.{png,pdf}` |

**DATA CELL — not run here.** Once notebooks 01–04 have produced real staged data and results:

```bash
python scripts/generate_original_study_figures.py
python scripts/generate_paper_placeholder_figures.py
python scripts/plot_post_covid_bootstrap.py --metric-family score
```

## Where this feeds back into the paper

The paper's own Results section (`my-local/weather-informed-modeling/paper/main.tex`) currently marks
every place a real number, table, or figure from this pipeline is still needed with a visible
`\TODO{...}` marker — see `my-local/weather-informed-modeling/prd/aug31_weather_paper_writing_plan.md`.
Running notebooks 01–05 end to end against real credentials and downloads is exactly what resolves
those TODOs: the CSVs and figures these scripts produce are the source of every number the Results
section is waiting on.